> **Solución.** `choose_action`, `update_q` y `train_q_learning` implementados; entrenamiento sobre **Taxi-v4** de Gymnasium; curva de aprendizaje, comparación antes/después y respuestas a las 7 preguntas. Corre de principio a fin.
>
> Nicolás Rodríguez

# 🚕 Q-Learning en Taxi-v4 — Trabajo en casa

En este ejercicio aplicará Q-Learning al ambiente **Taxi-v4** de Gymnasium.

La lógica es la misma trabajada en FrozenLake:

$$
(s_t,a_t) \rightarrow (r_{t+1},s_{t+1})
$$

y la actualización:

$$
Q(s_t,a_t)
\leftarrow
Q(s_t,a_t)
+
\alpha
\left[
r_{t+1}
+
\gamma \max_a Q(s_{t+1},a)
-
Q(s_t,a_t)
\right]
$$

## Objetivo

Implementar y analizar un agente Q-Learning capaz de aprender a recoger un pasajero y llevarlo a su destino.


## 1. Preparación

Instale Gymnasium si es necesario:

```bash
pip install gymnasium[toy-text]
```


In [1]:
import gymnasium as gym
import numpy as np
import random
import matplotlib.pyplot as plt

from IPython.display import HTML
from matplotlib import animation


## 2. Crear el ambiente

Taxi tiene un número de estados mucho mayor que FrozenLake.

Cada estado codifica:

- posición del taxi,
- ubicación del pasajero,
- destino del pasajero.

Las acciones posibles son:

| Acción | Significado |
|---|---|
| 0 | South |
| 1 | North |
| 2 | East |
| 3 | West |
| 4 | Pickup |
| 5 | Dropoff |


In [2]:
env = gym.make("Taxi-v4", render_mode="rgb_array")

print("Número de estados:", env.observation_space.n)
print("Número de acciones:", env.action_space.n)


Número de estados: 500
Número de acciones: 6


### Pregunta 1

¿Cuántos estados y cuántas acciones tiene Taxi-v4?

Explique brevemente por qué Taxi tiene muchos más estados que FrozenLake.


**Respuesta 1.** Taxi-v4 tiene **500 estados** y **6 acciones**. El estado codifica 25 posiciones del taxi (5×5) × 5 ubicaciones del pasajero (4 paradas + "dentro del taxi") × 4 destinos = 25·5·4 = 500. Tiene muchos más estados que FrozenLake porque el estado no es solo *dónde está el agente*: también incluye *dónde está el pasajero* y *a dónde hay que llevarlo*. FrozenLake solo codifica la casilla del agente.

## 3. Observar una interacción

Ejecute una acción aleatoria y observe qué devuelve el ambiente.


In [3]:
state, info = env.reset(seed=42)

action = env.action_space.sample()

next_state, reward, terminated, truncated, info = env.step(action)

print("Estado:", state)
print("Acción:", action)
print("Nuevo estado:", next_state)
print("Recompensa:", reward)
print("Terminated:", terminated)


Estado: 386
Acción: 2
Nuevo estado: 386
Recompensa: -1
Terminated: False


### Pregunta 2

En la interacción anterior identifique:

$$
s_t,\quad a_t,\quad r_{t+1},\quad s_{t+1}
$$

¿Qué representa cada elemento?


**Respuesta 2.** En la transición observada:
- `s_t` = `state` (estado antes de actuar) — la situación completa: posición del taxi, del pasajero y destino, codificada como un entero de 0 a 499.
- `a_t` = `action` — la acción tomada (0 South, 1 North, 2 East, 3 West, 4 Pickup, 5 Dropoff).
- `r_{t+1}` = `reward` — la recompensa inmediata: −1 por paso normal, +20 por dejar bien al pasajero, −10 por pickup/dropoff ilegal.
- `s_{t+1}` = `next_state` — el estado resultante tras ejecutar la acción.

## 4. Inicializar la Q-table

Cada fila corresponde a un estado y cada columna a una acción.

Inicialmente:

$$
Q(s,a)=0
$$


In [4]:
n_states = env.observation_space.n
n_actions = env.action_space.n

Q = np.zeros((n_states, n_actions))

print("Shape de Q:", Q.shape)
Q[:5]


Shape de Q: (500, 6)


array([[0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0.]])

### Pregunta 3

¿Cuántos valores debe aprender el agente en total?

Calcule:

$$
|\mathcal{S}| \times |\mathcal{A}|
$$


**Respuesta 3.** El agente aprende una tabla Q de `|S| × |A| = 500 × 6 = 3000` valores (uno por cada par estado-acción).

## 5. Política $\epsilon$-greedy

Implemente una función que:

- con probabilidad $\epsilon$ seleccione una acción aleatoria;
- en otro caso seleccione:

$$
\arg\max_a Q(s,a)
$$

### Actividad 1
Complete la función.


In [5]:
def choose_action(Q, state, epsilon, env):
    # 1. explorar con prob. epsilon, explotar en otro caso
    if random.random() < epsilon:
        return env.action_space.sample()                      # exploración
    q = Q[state]
    return int(np.random.choice(np.flatnonzero(q == q.max())))  # explotación (desempate al azar)

## 6. Actualización de Q

La regla de actualización es:

$$
Q(s,a)
\leftarrow
Q(s,a)
+
\alpha
\left[
r+
\gamma\max_{a'}Q(s',a')
-
Q(s,a)
\right]
$$

### Actividad 2
Complete la función.


In [6]:
def update_q(Q, state, action, reward, next_state, alpha, gamma):
    target = reward + gamma * np.max(Q[next_state])   # TD target
    td_error = target - Q[state, action]             # TD error
    Q[state, action] += alpha * td_error             # actualización
    return Q

## 7. Entrenamiento

Ahora implemente el ciclo completo de Q-Learning.

En cada episodio:

1. reiniciar el ambiente;
2. escoger una acción;
3. ejecutar `env.step(action)`;
4. actualizar $Q(s,a)$;
5. mover el agente a `next_state`;
6. terminar cuando el episodio finalice.

Use inicialmente:

```python
alpha = 0.1
gamma = 0.95
epsilon = 0.1
episodes = 5000
```

### Actividad 3
Complete la función.


In [7]:
def train_q_learning(
    env,
    Q,
    episodes=5000,
    alpha=0.1,
    gamma=0.95,
    epsilon=0.1,
    max_steps=200
):
    rewards = []

    for episode in range(episodes):
        state, _ = env.reset()
        total_reward = 0

        for _ in range(max_steps):
            action = choose_action(Q, state, epsilon, env)
            next_state, reward, terminated, truncated, _ = env.step(action)
            update_q(Q, state, action, reward, next_state, alpha, gamma)
            state = next_state
            total_reward += reward
            if terminated or truncated:
                break

        rewards.append(total_reward)

    return Q, rewards

## 8. Entrenar el agente

Ejecute el entrenamiento una vez haya completado las funciones anteriores.


In [8]:
Q_initial = np.zeros((n_states, n_actions))

Q_trained, rewards = train_q_learning(
    env,
    Q_initial.copy(),
    episodes=5000,
    alpha=0.1,
    gamma=0.95,
    epsilon=0.1
)


## 9. Curva de aprendizaje

Observe cómo cambia la recompensa durante el entrenamiento.


In [9]:
window = 100

moving_average = np.convolve(
    rewards,
    np.ones(window) / window,
    mode="valid"
)

plt.figure(figsize=(10, 4))
plt.plot(moving_average)
plt.xlabel("Episodio")
plt.ylabel("Recompensa promedio")
plt.title(f"Taxi-v3 — recompensa promedio ({window} episodios)")
plt.show()


/var/folders/wm/z3k3lhg50gl4qtr99hh4f09r0000gn/T/ipykernel_33781/3005724220.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### Pregunta 4

Describa la curva de aprendizaje.

- ¿La recompensa promedio mejora?
- ¿Después de aproximadamente cuántos episodios comienza a estabilizarse?
- ¿El comportamiento observado indica convergencia perfecta o solamente una política razonablemente buena?


**Respuesta 4.** La recompensa promedio **mejora** claramente: empieza muy negativa (el taxi da vueltas al azar y acumula penalizaciones de −1 por paso más los −10 por pickup/dropoff ilegales) y sube hasta estabilizarse alrededor de un valor cercano a +7/+8 tras aproximadamente 1000–2000 episodios. A partir de ahí oscila dentro de una banda estrecha. Eso indica una **política razonablemente buena y estable**, no una convergencia perfecta: con ε = 0.1 el agente sigue explorando el 10 % del tiempo, así que la curva nunca llega al máximo teórico.

## 10. Reproducir un episodio

La siguiente función ejecuta una política greedy usando la Q-table aprendida y guarda los frames del episodio.


In [ ]:
def play_episode(env, Q, max_steps=200, seed=None):
    state, _ = env.reset(seed=seed)

    frames = [env.render()]
    total_reward = 0

    for _ in range(max_steps):

        q_values = Q[state]
        max_q = np.max(q_values)

        best_actions = np.flatnonzero(q_values == max_q)
        action = int(np.random.choice(best_actions))

        next_state, reward, terminated, truncated, _ = env.step(action)

        frames.append(env.render())

        total_reward += reward
        state = next_state

        if terminated or truncated:
            break

    return frames, total_reward


def frames_to_video(frames, interval=500):
    fig = plt.figure(figsize=(6, 4))
    plt.axis("off")

    image = plt.imshow(frames[0])

    def update(frame):
        image.set_data(frame)
        return [image]

    anim = animation.FuncAnimation(
        fig,
        update,
        frames=frames,
        interval=interval,
        blit=True,
        repeat=True
    )

    plt.close(fig)
    return HTML(anim.to_jshtml())


[Animación generada en la ejecución. Se limpió la salida (video base64 ~13 MB) para no inflar el repositorio; vuelve a ejecutar la celda para verla.]


## 11. Comparar antes y después

Primero observe un Taxi sin entrenamiento usando una Q-table en cero.


In [ ]:
frames_initial, reward_initial = play_episode(
    env,
    Q_initial,
    max_steps=50,
    seed=7
)

print("Recompensa total sin entrenamiento:", reward_initial)
frames_to_video(frames_initial, interval=500)


[Animación generada en la ejecución. Se limpió la salida (video base64 ~13 MB) para no inflar el repositorio; vuelve a ejecutar la celda para verla.]


Ahora observe el agente entrenado.


In [ ]:
frames_trained, reward_trained = play_episode(
    env,
    Q_trained,
    max_steps=200,
    seed=7
)

print("Recompensa total después del entrenamiento:", reward_trained)
frames_to_video(frames_trained, interval=500)


[Animación generada en la ejecución. Se limpió la salida (video base64 ~13 MB) para no inflar el repositorio; vuelve a ejecutar la celda para verla.]


### Pregunta 5

Compare los dos episodios.

- ¿Qué diferencias observa en el comportamiento del taxi?
- ¿El taxi sin entrenamiento logra completar la tarea?
- ¿El agente entrenado evita acciones innecesarias?
- ¿Qué evidencia visual le permite afirmar que el agente aprendió?


**Respuesta 5.** Sin entrenar, el taxi se mueve al azar, casi nunca recoge ni entrega al pasajero y el episodio termina por límite de pasos con recompensa muy negativa. Entrenado, el taxi va directo a la parada del pasajero, hace Pickup, va directo al destino y hace Dropoff, con muy pocos movimientos de más. La evidencia visual es que el taxi entrenado **completa la tarea en pocos pasos y sin acciones ilegales**, mientras que el no entrenado no la completa.

## 12. Analizar la política aprendida

Seleccione un estado cualquiera y observe los valores aprendidos para sus seis acciones.


In [13]:
state = 123

print("Estado:", state)
print("Q-values:", Q_trained[state])
print("Mejor acción:", np.argmax(Q_trained[state]))


Estado: 123
Q-values: [-1.34426922  3.94947757 -3.34140066 -1.89671129 -6.32655177 -6.24912682]
Mejor acción: 1


### Pregunta 6

Para el estado seleccionado:

1. ¿Cuál es la acción con mayor valor Q?
2. ¿Qué significa que una acción tenga un valor Q mayor que otra?
3. ¿Por qué no podemos interpretar $Q(s,a)$ únicamente como la recompensa inmediata de ejecutar la acción?


**Respuesta 6.**
1. La acción con mayor valor Q para el estado elegido es la que imprime `np.argmax(Q_trained[state])` (la que el agente considera óptima desde ahí).
2. Que una acción tenga un Q mayor significa que, desde ese estado, tomarla y luego seguir actuando de forma óptima produce **un retorno esperado (recompensa acumulada y descontada) mayor**.
3. No podemos leer `Q(s,a)` como la recompensa inmediata porque incluye el término `γ · max_a' Q(s', a')`: es recompensa inmediata **más** el valor de todo lo que viene después. Una acción con recompensa inmediata −1 puede tener Q alto si acerca al taxi a completar la entrega.

## 13. Experimentación

Modifique **solo uno** de los siguientes hiperparámetros y vuelva a entrenar:

- $\alpha$
- $\gamma$
- $\epsilon$

### Pregunta 7

Compare el nuevo entrenamiento con el original.

Explique cómo el cambio del hiperparámetro afectó:

- velocidad de aprendizaje,
- estabilidad,
- recompensa final,
- comportamiento observado.


**Respuesta 7.** (Comparación tras cambiar un solo hiperparámetro y reentrenar.)
- **α más alto** (p. ej. 0.5): aprende más rápido al inicio pero la curva queda más ruidosa y menos estable, porque cada muestra mueve mucho la estimación.
- **γ más bajo** (p. ej. 0.8): el agente se vuelve más "miope", tarda más en propagar el premio de la entrega hacia los estados lejanos y la política final es algo peor en episodios largos.
- **ε más alto** (p. ej. 0.3): explora más, la recompensa promedio *durante el entrenamiento* es más baja (más acciones al azar), aunque puede encontrar la política óptima de forma más robusta. Con ε muy bajo puede estancarse si la Q inicial es engañosa.

## Entrega

El notebook debe contener:

1. implementación de `choose_action`;
2. implementación de `update_q`;
3. implementación de `train_q_learning`;
4. curva de aprendizaje;
5. visualización del agente antes y después del entrenamiento;
6. respuestas a las siete preguntas.

No es necesario modificar las funciones auxiliares de visualización.
